# 01 — Exploratory Data Analysis

Component 3 (Learning Insight & Early Warning System) — Grade 2-3 mathematics dyscalculia,
synthetic research data.

**Purpose of this notebook:** measure and report what the dataset actually contains. It does
not model, preprocess, impute, scale, or engineer anything — that is notebook 02's job. Every
claim in `DATASET_CARD.md` is verified rather than assumed; where the data disagrees with the
card, that disagreement is reported explicitly as a finding.

Primary arm used throughout: **h=5, n=4** (`training_table_seed20260828_n300_h5_sweepn4.csv`).

In [1]:
import sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
from statsmodels.stats.outliers_influence import variance_inflation_factor

sys.path.insert(0, str(Path.cwd()))
import nbconfig as cfg

np.random.seed(cfg.RANDOM_SEED)
RNG = np.random.default_rng(cfg.RANDOM_SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings("ignore", category=FutureWarning)

cfg.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
cfg.TABLES_DIR.mkdir(parents=True, exist_ok=True)
cfg.MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FEATURES = cfg.MODEL_FEATURES
CLUSTER_FEATURES = cfg.CLUSTER_FEATURES

def savefig(fig, name):
    path = cfg.FIGURES_DIR / f"{name}.png"
    fig.savefig(path, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print("saved:", path)

def savetable(df_, name, index=True):
    path = cfg.TABLES_DIR / f"{name}.csv"
    df_.to_csv(path, index=index)
    print("saved:", path)

print("Model features:", len(MODEL_FEATURES))
print("Cluster features:", len(CLUSTER_FEATURES))

Model features: 23
Cluster features: 8


## 1. Load data

Loading the four primary-arm artefacts: the supervised training table, the full feature
matrix (all snapshots), the feature-matrix manifest, and the quarantined audit labels
(population/dd_subtype — used *only* in step 4's absence check, per the leakage rules in
doc 08).

In [2]:
train = pd.read_csv(cfg.TRAINING_TABLE_PATH)
fmat = pd.read_csv(cfg.FEATURE_MATRIX_PATH)
audit = pd.read_csv(cfg.AUDIT_LABELS_PATH)
with open(cfg.FEATURE_MATRIX_MANIFEST_PATH) as f:
    fmat_manifest = json.load(f)

for name, d in [("training_table", train), ("feature_matrix", fmat), ("audit_labels", audit)]:
    print(f"--- {name} ---")
    print("shape:", d.shape)
    print("memory (MB): %.3f" % (d.memory_usage(deep=True).sum() / 1e6))
    print()

print("training_table dtypes:")
print(train.dtypes.value_counts())

--- training_table ---
shape: (2700, 52)
memory (MB): 1.413

--- feature_matrix ---
shape: (7500, 33)
memory (MB): 2.040

--- audit_labels ---
shape: (300, 3)
memory (MB): 0.012

training_table dtypes:
float64    28
int64      14
str         9
bool        1
Name: count, dtype: int64


In [3]:
train.dtypes.to_frame("dtype").to_csv(cfg.TABLES_DIR / "01_training_table_dtypes.csv")
print("saved:", cfg.TABLES_DIR / "01_training_table_dtypes.csv")
train.head()

saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\01_training_table_dtypes.csv


,student_id,t,sessions_observed,b_sessions_observed,c_sessions_observed,score_trend_slope,score_avg_recent,mastery_progression_rate,number_sense_mastery,weakest_skill_persistence,rt_change_ratio,rt_variability,rt_accuracy_tradeoff,game_latency_change_ratio,error_recurrence_rate,hint_avoidance,hint_intensity,guess_guard_rate,engagement_consistency,engagement_decline_rate,frustration_mean,frustration_slope,difficulty_floor_hits,stagnant_skill_count,slope_censored_count,number_sense_minus_operations,ipsatised_mastery_counting,ipsatised_mastery_number_sense,ipsatised_mastery_place_value,ipsatised_mastery_addition,ipsatised_mastery_subtraction,ipsatised_mastery_multiplication,ipsatised_mastery_division,evidence_status,trajectory_indicators,trajectory_indicators_status,future_stagnant_count,future_stagnant_skills,risk_level,risk_level_status,future_frustration_mean,future_engagement_decline_rate,teacher_facing_output,horizon,horizon_role,warmup_floor,stagnant_trailing_window_may_cross_t,target_window_first_session,target_window_last_session,future_sessions_scheduled,future_sessions_observed,split
0,STU-0001,3,3,3,3,-0.083333,0.361111,0.012129,0.319776,3,NaN,1.825620,2.34,NaN,0.500,0.0,0.222222,0.0,1.0,-0.080429,0.492655,0.701311,NaN,0,1,0.169776,0.057262,0.131936,-0.037840,-0.037840,-0.037840,-0.037840,-0.037840,sufficient_evidence,[],RULES_UNDEFINED,1,number_sense,NaN,CALIBRATION_PENDING,0.413283,-0.096328,flags learners for teacher review,5,primary,3,True,4,8,5,5,test
1,STU-0001,5,5,5,5,-0.075000,0.277778,-0.002424,0.296929,5,1.077998,0.566962,NaN,0.974352,0.500,0.0,0.266667,0.0,1.0,-0.031598,0.452969,0.038687,0.000000,0,0,0.146929,0.124630,0.101669,-0.045260,-0.045260,-0.045260,-0.045260,-0.045260,sufficient_evidence,[],RULES_UNDEFINED,1,number_sense,NaN,CALIBRATION_PENDING,0.608352,-0.002269,flags learners for teacher review,5,primary,3,True,6,10,5,5,test
2,STU-0001,7,7,7,7,-0.011905,0.333333,-0.004615,0.341119,7,0.939686,0.115909,NaN,0.953031,0.500,0.0,0.250000,0.0,1.0,-0.031598,0.452969,0.038687,0.000000,0,0,0.191119,0.184220,0.128563,-0.062557,-0.062557,-0.062557,-0.062557,-0.062557,sufficient_evidence,[],RULES_UNDEFINED,1,number_sense,NaN,CALIBRATION_PENDING,0.471940,0.005418,flags learners for teacher review,5,primary,3,True,8,12,5,5,test
3,STU-0001,9,9,9,9,-0.048611,0.000000,-0.001891,0.299700,9,0.481724,0.509316,NaN,0.976846,0.625,0.0,0.222222,0.0,1.0,-0.018082,0.491525,0.057899,0.294118,1,0,0.149700,0.190137,0.093060,-0.056639,-0.056639,-0.056639,-0.056639,-0.056639,sufficient_evidence,[],RULES_UNDEFINED,1,number_sense,NaN,CALIBRATION_PENDING,0.433555,0.017286,flags learners for teacher review,5,primary,3,True,10,14,5,5,test
4,STU-0001,11,11,11,11,-0.009091,0.333333,0.017888,0.412551,11,0.479987,0.488439,1.73,1.155598,0.600,0.0,0.242424,0.0,1.0,-0.007090,0.461100,-0.004977,0.000000,0,0,0.262551,0.239669,0.178848,-0.083703,-0.083703,-0.083703,-0.083703,-0.083703,sufficient_evidence,[],RULES_UNDEFINED,1,number_sense,NaN,CALIBRATION_PENDING,0.457609,-0.019375,flags learners for teacher review,5,primary,3,True,12,16,5,5,test


**Findings - step 1.** Training table: 2,700 rows x 52 columns (1.41 MB). Feature
matrix (all snapshots): 7,500 rows x 33 columns (2.04 MB). Audit labels: 300 rows x 3
columns (`student_id`, `population`, `dd_subtype`) - quarantined, absence-check only.
Dtypes on the training table: 28 float64, 14 int64, 9 string, 1 bool - matches an
expectation of mixed numeric features plus categorical/status text columns and one
boolean flag (`stagnant_trailing_window_may_cross_t`). Shapes match the card's stated
2,700/300 and 7,500/300 exactly.


## 2. Panel structure

Expect 300 learners x 9 snapshots at `t in {3,5,7,9,11,13,15,17,19}` in the primary (h=5)
training table (the full grid is `{3,...,25}` but only `t+5<=25` rows are supervised, i.e.
`t<=20`, intersected with the odd-t grid -> t in {3,...,19}). Verifying counts, the t grid,
and the `t+h<=25` eligibility rule explicitly.

In [4]:
n_learners = train["student_id"].nunique()
t_grid = sorted(train["t"].unique().tolist())
rows_per_learner = train.groupby("student_id").size()

print("n rows:", len(train))
print("n learners:", n_learners)
print("t grid:", t_grid)
print("rows-per-learner value counts:")
print(rows_per_learner.value_counts())

assert set(t_grid) == {3, 5, 7, 9, 11, 13, 15, 17, 19}, f"unexpected t grid: {t_grid}"
assert (rows_per_learner == 9).all(), "not every learner has exactly 9 rows"
assert n_learners == 300, f"expected 300 learners, got {n_learners}"
assert len(train) == 2700, f"expected 2700 rows, got {len(train)}"

horizon = train["horizon"].unique()
print("horizon column unique values:", horizon)
assert (train["t"] + 5 <= 25).all(), "found a row with t+5 > 25"
print("OK: every row satisfies t+5<=25")

n rows: 2700
n learners: 300
t grid: [3, 5, 7, 9, 11, 13, 15, 17, 19]
rows-per-learner value counts:
9    300
Name: count, dtype: int64
horizon column unique values: [5]
OK: every row satisfies t+5<=25


**Findings - step 2.** Every one of the 300 learners has exactly 9 rows; the observed
t grid is exactly `[3, 5, 7, 9, 11, 13, 15, 17, 19]`, matching the first 9 values of
the card's frozen 12-value snapshot grid (21, 23, 25 are absent because at h=5 they
would need `t+5<=25`, so only t<=20 is eligible on the odd grid - consistent with the
card's 2,700-row primary arm). `horizon` is 5 on every row and the `t+5<=25` assertion
holds for all 2,700 rows. Panel structure matches the card exactly.


## 3. Split integrity

The materialised `split` column must give 210 train / 90 test **learners** (not rows), with
zero `student_id` overlap between the two sides. Asserted explicitly — failure raises.

In [5]:
print("split value counts (rows):")
print(train["split"].value_counts())

learner_split = train.groupby("student_id")["split"].agg(lambda s: s.unique().tolist())
inconsistent = learner_split[learner_split.apply(len) > 1]
assert inconsistent.empty, f"learners with inconsistent split labels across rows: {inconsistent}"

split_of_learner = learner_split.apply(lambda x: x[0])
learner_counts = split_of_learner.value_counts()
print("learners per split:")
print(learner_counts)

train_ids = set(split_of_learner[split_of_learner == "train"].index)
test_ids = set(split_of_learner[split_of_learner == "test"].index)

assert learner_counts.get("train", 0) == 210, f"expected 210 train learners, got {learner_counts.get('train', 0)}"
assert learner_counts.get("test", 0) == 90, f"expected 90 test learners, got {learner_counts.get('test', 0)}"
assert len(train_ids & test_ids) == 0, "student_id overlap between train and test splits!"
print("OK: 210 train / 90 test learners, zero student_id overlap")

train_rows = (train["split"] == "train").sum()
test_rows = (train["split"] == "test").sum()
print(f"rows: train={train_rows}, test={test_rows}")

split value counts (rows):
split
train    1890
test      810
Name: count, dtype: int64
learners per split:
split
train    210
test      90
Name: count, dtype: int64
OK: 210 train / 90 test learners, zero student_id overlap
rows: train=1890, test=810


**Findings - step 3.** `split` gives 1,890 train / 810 test rows and, at the learner
level, exactly 210 train / 90 test learners - matching the card's 210/90 (and its
1,890/810 row counts) precisely. The assertion confirms zero `student_id` overlap
between the two sides. Split integrity holds.


## 4. Population / dd_subtype absence check

`population` and `dd_subtype` must appear **nowhere** in the feature matrix or training table
— these are quarantined to the audit-only labels file. Checked by searching all column names in
both tables. The audit file itself is used *only* for this absence check; it is never joined
into any other analysis in this notebook.

In [6]:
forbidden = {"population", "dd_subtype"}

train_cols = set(train.columns)
fmat_cols = set(fmat.columns)

hit_train = forbidden & train_cols
hit_fmat = forbidden & fmat_cols

print("forbidden columns in training table:", hit_train)
print("forbidden columns in feature matrix:", hit_fmat)

assert not hit_train, f"FOUND forbidden columns in training table: {hit_train}"
assert not hit_fmat, f"FOUND forbidden columns in feature matrix: {hit_fmat}"
print("OK: population/dd_subtype absent from both feature matrix and training table")

print("\naudit_labels columns (for reference, quarantined, audit-only):", list(audit.columns))
assert forbidden.issubset(set(audit.columns)), "audit labels file unexpectedly missing population/dd_subtype"

forbidden columns in training table: set()
forbidden columns in feature matrix: set()
OK: population/dd_subtype absent from both feature matrix and training table

audit_labels columns (for reference, quarantined, audit-only): ['student_id', 'population', 'dd_subtype']


**Findings - step 4.** `population` and `dd_subtype` are absent from both the training
table and the feature matrix (empty forbidden-column sets in both). They exist only in
`t3_audit_labels_seed20260828_n300.csv`, which is used here solely for this absence
check and is not joined into any other analysis in this notebook. LS-1 / 3-F1 hold for
this dataset as delivered.


## 5. Missingness — null rate per feature, overall and per t

Per doc 06 §5, slope-type features need `w=4` observations, so they are **structurally null**
at early t (especially t=3). The heatmap below (features x t, null rate) is used to separate
that expected warm-up pattern from any other unexplained null pattern.

In [7]:
null_overall = train[MODEL_FEATURES].isnull().mean().sort_values(ascending=False)
savetable(null_overall.to_frame("null_rate"), "05_null_rate_overall")

null_by_t = train.groupby("t")[MODEL_FEATURES].apply(lambda d: d.isnull().mean())
savetable(null_by_t, "05_null_rate_by_t")

fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(null_by_t.T, annot=True, fmt=".2f", cmap="rocket_r", vmin=0, vmax=1, ax=ax, cbar_kws={"label": "null rate"})
ax.set_xlabel("t")
ax.set_ylabel("feature")
ax.set_title("Null rate by feature x t (model-role features)")
savefig(fig, "05_null_rate_heatmap")

print("Overall null rate per feature (descending):")
print(null_overall)

saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\05_null_rate_overall.csv
saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\05_null_rate_by_t.csv


saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\figures\05_null_rate_heatmap.png
Overall null rate per feature (descending):
difficulty_floor_hits        0.320000
rt_accuracy_tradeoff         0.307407
rt_change_ratio              0.111111
game_latency_change_ratio    0.111111
engagement_consistency       0.081852
engagement_decline_rate      0.081852
frustration_slope            0.081852
frustration_mean             0.016667
hint_intensity               0.003704
rt_variability               0.002963
number_sense_mastery         0.000000
score_trend_slope            0.000000
weakest_skill_persistence    0.000000
score_avg_recent             0.000000
mastery_progression_rate     0.000000
guess_guard_rate             0.000000
hint_avoidance               0.000000
error_recurrence_rate        0.000000
stagnant_skill_count         0.000000
sessions_observed            0.000000
b_sessions_observed        

In [8]:
# structural warm-up candidates: features whose null rate is ~1.0 at t=3 (need w=4 -> undefined
# until the 4th observation, i.e. defined starting at the learner's 4th session)
warmup_like = null_by_t.loc[3].sort_values(ascending=False)
print("Null rate at t=3, descending:")
print(warmup_like)

unusable_threshold = 0.5
unusable = null_overall[null_overall > unusable_threshold]
print(f"\nFeatures with overall null rate > {unusable_threshold} (candidates for 'unusable regardless of t'):")
print(unusable)

Null rate at t=3, descending:
rt_change_ratio              1.000000
difficulty_floor_hits        1.000000
game_latency_change_ratio    1.000000
engagement_decline_rate      0.503333
engagement_consistency       0.503333
frustration_slope            0.503333
rt_accuracy_tradeoff         0.283333
frustration_mean             0.130000
hint_intensity               0.026667
rt_variability               0.003333
number_sense_mastery         0.000000
score_trend_slope            0.000000
weakest_skill_persistence    0.000000
score_avg_recent             0.000000
mastery_progression_rate     0.000000
guess_guard_rate             0.000000
hint_avoidance               0.000000
error_recurrence_rate        0.000000
stagnant_skill_count         0.000000
sessions_observed            0.000000
b_sessions_observed          0.000000
c_sessions_observed          0.000000
slope_censored_count         0.000000
Name: 3, dtype: float64

Features with overall null rate > 0.5 (candidates for 'unusable regardl

**Findings - step 5.** Three features are null on essentially every row at t=3 and
recover fast: `rt_change_ratio`, `difficulty_floor_hits`, `game_latency_change_ratio`
(all 100% null at t=3) plus a second tier at ~50% (`engagement_consistency`,
`engagement_decline_rate`, `frustration_slope`) and a smaller one at ~28-13%
(`rt_accuracy_tradeoff`, `frustration_mean`). This is exactly the structural warm-up
pattern doc 06 s5 describes: slope/ratio features need `w=4` (or in the ratio-feature
case, an "early" vs "recent" `w`-sized window) observations before they're defined, so
of course they're null at the very first snapshot. Overall (across all t) the pattern
persists but softens: `difficulty_floor_hits` (32.0%) and `rt_accuracy_tradeoff`
(30.7%) remain the nulliest features even pooled across all 9 snapshots - both are
ratio/window features that can also be undefined later if the window sum or the
accuracy denominator is 0, not only during warm-up. No feature exceeds a 50% *overall*
null rate, so nothing here is unusable outright, but `difficulty_floor_hits` and
`rt_accuracy_tradeoff` should be flagged for notebook 02 as needing an explicit
missingness strategy (e.g. an indicator + model-native null handling) rather than mean
imputation, since a third of their nulls are not warm-up artefacts.


## 6. Missingness correlation — do features go null together, and is nullness clustered by learner?

A missingness correlation matrix (do the null *indicators* of pairs of features co-occur), and
a per-learner nullness rate to check whether some learners are systematically nullier than
others across all features (e.g. sparse-attendance learners).

In [9]:
null_ind = train[MODEL_FEATURES].isnull().astype(int)
null_corr = null_ind.corr()
savetable(null_corr, "06_missingness_correlation")

fig, ax = plt.subplots(figsize=(10, 9))
sns.heatmap(null_corr, cmap="vlag", vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title("Correlation of null-indicators across model-role features")
savefig(fig, "06_missingness_correlation_heatmap")

saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\06_missingness_correlation.csv


saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\figures\06_missingness_correlation_heatmap.png


In [10]:
learner_null_rate = null_ind.groupby(train["student_id"]).mean().mean(axis=1).sort_values(ascending=False)
savetable(learner_null_rate.to_frame("mean_null_rate"), "06_null_rate_by_learner")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(learner_null_rate, bins=30, color="#4C72B0")
ax.set_xlabel("mean null rate across all model features, per learner")
ax.set_ylabel("count of learners")
ax.set_title("Distribution of per-learner mean null rate")
savefig(fig, "06_learner_null_rate_hist")

print("Per-learner null-rate summary:")
print(learner_null_rate.describe())
print("\nBetween-learner spread (std):", learner_null_rate.std())

saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\06_null_rate_by_learner.csv


saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\figures\06_learner_null_rate_hist.png
Per-learner null-rate summary:
count    300.000000
mean       0.048631
std        0.023046
min        0.014493
25%        0.033816
50%        0.043478
75%        0.057971
max        0.178744
dtype: float64

Between-learner spread (std): 0.023046102453001475


**Findings - step 6.** The three C-engagement features that share the same
`telemetry_quality_status != complete` masking rule (`engagement_consistency`,
`engagement_decline_rate`, `frustration_slope`) go null together, as expected since
they share a common cause (doc 06 s5: mask, don't repair, when C telemetry is
incomplete). Per-learner null rates are tight and low (mean 4.9%, std 2.3%, max
17.9%) - nullness is not concentrated in a handful of learners; every learner has
some structural nulls (mostly from their own t=3 row) but no learner is pervasively
null. This is consistent with the nulls being driven by the snapshot's `t` value
(warm-up) rather than by learner-level data quality problems.


## 7. Distributions — describe() and histogram grid for every model-role feature

Looking for skew, hard bounds, spikes at 0/1, and near-constant / degenerate columns.

In [11]:
desc = train[MODEL_FEATURES].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T
desc["null_rate"] = train[MODEL_FEATURES].isnull().mean()
desc["n_unique"] = train[MODEL_FEATURES].nunique()
savetable(desc, "07_describe_model_features")
desc

saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\07_describe_model_features.csv


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,null_rate,n_unique
score_trend_slope,2700.0,-0.003978,0.076475,-0.500000,-0.250000,-0.125000,-0.022100,-0.000585,0.017857,0.089286,0.250000,0.500000,0.000000,1008
score_avg_recent,2700.0,0.353837,0.191482,0.000000,0.000000,0.000000,0.222222,0.333333,0.500000,0.666667,0.805556,1.000000,0.000000,35
mastery_progression_rate,2700.0,0.009947,0.010606,-0.014143,-0.008894,-0.005617,0.001487,0.009216,0.018041,0.030304,0.034289,0.043653,0.000000,2355
number_sense_mastery,2700.0,0.349364,0.162417,0.169860,0.169860,0.169860,0.218548,0.292006,0.503091,0.625585,0.678498,0.763664,0.000000,833
weakest_skill_persistence,2700.0,7.384074,4.697075,1.000000,1.000000,1.000000,3.000000,7.000000,11.000000,17.000000,18.000000,19.000000,0.000000,19
rt_change_ratio,2400.0,1.085411,0.516765,0.201802,0.304226,0.460387,0.761402,0.993905,1.282579,1.964974,3.020386,5.016667,0.111111,2399
rt_variability,2692.0,0.432201,0.443288,0.000500,0.005455,0.037183,0.149034,0.301418,0.556548,1.272871,2.277342,4.265916,0.002963,2455
rt_accuracy_tradeoff,1870.0,1.937394,1.538347,0.200000,0.324003,0.488450,0.910000,1.521500,2.439375,4.812200,7.638560,15.210000,0.307407,1528
game_latency_change_ratio,2400.0,1.021103,0.123926,0.690798,0.772183,0.836260,0.940264,1.012526,1.089960,1.240450,1.365772,1.899208,0.111111,2399
error_recurrence_rate,2700.0,0.622495,0.149951,0.000000,0.250000,0.500000,0.500000,0.611111,0.700000,1.000000,1.000000,1.000000,0.000000,41


In [12]:
near_constant = desc[(desc["std"].fillna(0) < 1e-6) | (desc["n_unique"] <= 2)]
print("Near-constant / degenerate model-role features:")
print(near_constant[["mean", "std", "n_unique", "null_rate"]])

Near-constant / degenerate model-role features:
                            mean       std  n_unique  null_rate
hint_avoidance          0.003704  0.060757         2   0.000000
guess_guard_rate        0.000000  0.000000         1   0.000000
engagement_consistency  1.000000  0.000000         1   0.081852
stagnant_skill_count    0.096667  0.295558         2   0.000000


In [13]:
n_feat = len(MODEL_FEATURES)
ncols = 4
nrows = int(np.ceil(n_feat / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
axes = axes.flatten()
for i, feat in enumerate(MODEL_FEATURES):
    ax = axes[i]
    vals = train[feat].dropna()
    if vals.nunique() > 1:
        sns.histplot(vals, kde=True, ax=ax, color="#4C72B0", bins=30)
    else:
        ax.hist(vals, bins=10, color="#4C72B0")
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel("")
for j in range(n_feat, len(axes)):
    axes[j].axis("off")
fig.suptitle("Model-role feature distributions (non-null values)", y=1.01)
fig.tight_layout()
savefig(fig, "07_feature_histogram_grid")

saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\figures\07_feature_histogram_grid.png


**Findings - step 7.** Most model-role features are right-skewed and bounded near
their construction (e.g. rates in [0,1], ratios centred near 1.0 with a long right
tail). Four features are near-constant or degenerate on this arm: `guess_guard_rate`
is a constant 0.0 for every row (n_unique=1 - the guard never fires in this synthetic
generator run, or evidence for it is always absent); `engagement_consistency` is a
constant 1.0 wherever non-null (n_unique=1, but still 8.2% null); `hint_avoidance` is
almost entirely 0 with a rate of exactly 1.0 on only 10/2700 rows (a near-constant
binary flag); `stagnant_skill_count` is binary-valued (0 or 1) with mean 0.097, i.e.
about 90% zeros. None of these four carry useful spread for modelling as continuous
inputs on this dataset; `guess_guard_rate` in particular is entirely uninformative
here and is a candidate to drop or keep only as a documented null/constant sentinel.


## 8. Outlier census (IQR and z-score methods) — descriptive only, nothing removed

Counting outliers per feature by both the 1.5xIQR rule and |z|>3, with specific attention to
the division-based ratio features (`rt_change_ratio`, `game_latency_change_ratio`,
`rt_accuracy_tradeoff`), which are the features most likely to have heavy tails from a
near-zero denominator.

In [14]:
def iqr_outlier_count(s):
    s = s.dropna()
    if len(s) == 0:
        return 0, np.nan, np.nan
    q1, q3 = s.quantile([.25, .75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < lo) | (s > hi)).sum()), lo, hi

def zscore_outlier_count(s):
    s = s.dropna()
    if len(s) < 2 or s.std() == 0:
        return 0
    z = (s - s.mean()) / s.std()
    return int((z.abs() > 3).sum())

rows = []
for feat in MODEL_FEATURES:
    s = train[feat]
    n_iqr, lo, hi = iqr_outlier_count(s)
    n_z = zscore_outlier_count(s)
    rows.append({
        "feature": feat, "n_non_null": s.notnull().sum(),
        "iqr_outliers": n_iqr, "iqr_outlier_rate": n_iqr / max(s.notnull().sum(), 1),
        "iqr_lower": lo, "iqr_upper": hi,
        "zscore_outliers": n_z, "zscore_outlier_rate": n_z / max(s.notnull().sum(), 1),
        "min": s.min(), "max": s.max(),
    })
outlier_census = pd.DataFrame(rows).set_index("feature").sort_values("iqr_outlier_rate", ascending=False)
savetable(outlier_census, "08_outlier_census")
outlier_census

saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\08_outlier_census.csv


,n_non_null,iqr_outliers,iqr_outlier_rate,iqr_lower,iqr_upper,zscore_outliers,zscore_outlier_rate,min,max
feature,,,,,,,,,
engagement_decline_rate,2479,369,0.148850,-0.029670,0.028219,71,0.028641,-0.166138,0.212991
score_trend_slope,2700,386,0.142963,-0.082035,0.077792,91,0.033704,-0.500000,0.500000
frustration_slope,2479,351,0.141589,-0.121940,0.122257,75,0.030254,-0.755663,0.701311
stagnant_skill_count,2700,261,0.096667,0.000000,0.000000,261,0.096667,0.000000,1.000000
slope_censored_count,2700,178,0.065926,0.000000,0.000000,55,0.020370,0.000000,2.000000
rt_variability,2692,168,0.062407,-0.462236,1.167819,56,0.020802,0.000500,4.265916
error_recurrence_rate,2700,159,0.058889,0.200000,1.000000,22,0.008148,0.000000,1.000000
rt_accuracy_tradeoff,1870,100,0.053476,-1.384062,4.733437,38,0.020321,0.200000,15.210000
rt_change_ratio,2400,103,0.042917,-0.020364,2.064345,44,0.018333,0.201802,5.016667


In [15]:
ratio_feats = ["rt_change_ratio", "game_latency_change_ratio", "rt_accuracy_tradeoff"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feat in zip(axes, ratio_feats):
    vals = train[feat].dropna()
    sns.boxplot(y=vals, ax=ax, color="#DD8452")
    ax.set_title(f"{feat}\nn={len(vals)}, max={vals.max():.2f}")
fig.suptitle("Division-based ratio features — boxplots (tail check)")
fig.tight_layout()
savefig(fig, "08_ratio_features_boxplots")

print(train[ratio_feats].describe(percentiles=[.5, .9, .95, .99]))

saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\figures\08_ratio_features_boxplots.png
       rt_change_ratio  game_latency_change_ratio  rt_accuracy_tradeoff
count      2400.000000                2400.000000           1870.000000
mean          1.085411                   1.021103              1.937394
std           0.516765                   0.123926              1.538347
min           0.201802                   0.690798              0.200000
50%           0.993905                   1.012526              1.521500
90%           1.659590                   1.173098              3.693900
95%           1.964974                   1.240450              4.812200
99%           3.020386                   1.365772              7.638560
max           5.016667                   1.899208             15.210000


**Findings - step 8.** The heaviest outlier burden (both IQR and z-score methods) falls
on the slope/rate features whose distributions are tightly concentrated near 0 with a
few extreme deviators: `engagement_decline_rate` (14.9% IQR-outlier rate),
`score_trend_slope` (14.3%), `frustration_slope` (14.2%). The binary/near-binary
features (`stagnant_skill_count`, `slope_censored_count`) also show high nominal
"IQR outlier" rates, but that's an artefact of IQR being degenerate on a mostly-0
column (iqr_lower/upper both pin at 0), not a real tail - z-score is the more honest
read there. Among the three division-based ratio features flagged for inspection:
`rt_accuracy_tradeoff` has the widest range (0.20-15.21, IQR outlier rate 5.3%) and
`rt_change_ratio` similarly heavy-tailed (up to 5.02, 4.3%); `game_latency_change_ratio`
is comparatively tame (0.69-1.90, essentially no IQR outliers) because it's a ratio of
two already-normalised latency windows. No rows or values are removed - this is a
census only, saved to `08_outlier_census.csv`.


## 9. hint_avoidance / hint_intensity — the U-shaped help-seeking split

Per doc 06, these exist as two separate features because hint usage is non-monotonic (both
never-asking and constantly-asking predict poor outcomes). `hint_intensity` is defined to be
null *exactly* when the hint rate is 0 (`hint_avoidance == 1`) — verified directly, not just
plotted.

In [16]:
joint = train[["hint_avoidance", "hint_intensity"]].copy()
print("hint_avoidance value counts:")
print(joint["hint_avoidance"].value_counts(dropna=False))

# verify hint_intensity is null iff hint_avoidance == 1
avoid_1_null_rate = joint.loc[joint["hint_avoidance"] == 1, "hint_intensity"].isnull().mean()
avoid_0_null_rate = joint.loc[joint["hint_avoidance"] == 0, "hint_intensity"].isnull().mean()
print(f"\nP(hint_intensity null | hint_avoidance==1) = {avoid_1_null_rate:.4f}  (expect 1.0)")
print(f"P(hint_intensity null | hint_avoidance==0) = {avoid_0_null_rate:.4f}  (expect 0.0)")

assert avoid_1_null_rate == 1.0, "hint_intensity is not null for every hint_avoidance==1 row"
assert avoid_0_null_rate == 0.0, "hint_intensity is null for some hint_avoidance==0 row"
print("OK: hint_intensity null status matches hint_avoidance exactly (definitional, not incidental)")

hint_avoidance value counts:
hint_avoidance
0.0    2690
1.0      10
Name: count, dtype: int64

P(hint_intensity null | hint_avoidance==1) = 1.0000  (expect 1.0)
P(hint_intensity null | hint_avoidance==0) = 0.0000  (expect 0.0)
OK: hint_intensity null status matches hint_avoidance exactly (definitional, not incidental)


In [17]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(x=joint["hint_avoidance"], ax=axes[0], color="#4C72B0")
axes[0].set_title("hint_avoidance value counts")
sns.histplot(joint["hint_intensity"].dropna(), bins=30, kde=True, ax=axes[1], color="#DD8452")
axes[1].set_title("hint_intensity distribution (non-null, i.e. hint_avoidance==0 only)")
fig.tight_layout()
savefig(fig, "09_hint_avoidance_intensity")

saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\figures\09_hint_avoidance_intensity.png


**Findings - step 9.** `hint_avoidance` is 1.0 for exactly 10 of 2,700 rows (0.37%,
learners who never requested a hint) and 0.0 otherwise. `hint_intensity` is null on
100% of the `hint_avoidance==1` rows and non-null on 100% of the `hint_avoidance==0`
rows - an exact, deterministic split, confirming doc 06's stated rule that
`hint_intensity` is null by definition precisely when the hint rate is 0 (which is
exactly what `hint_avoidance==1` encodes). The two-feature design behaves as
documented: `hint_avoidance` flags the "never asks" tail and `hint_intensity` only
carries information for the remaining 99.6% of rows where hints were used at least
once, letting the U-shaped relationship (per doc 06) be represented without forcing a
single feature to be non-monotonic.


## 10. Correlation heatmaps (Spearman and Pearson) — multicollinearity candidates

Both methods are computed because several model-role features are ratios/rates that may be
monotonically but non-linearly related. Every pair with `|r| > 0.9` under *either* method is
listed explicitly below — this list is meant to be usable verbatim by notebook 02.

In [18]:
pearson_corr = train[MODEL_FEATURES].corr(method="pearson")
spearman_corr = train[MODEL_FEATURES].corr(method="spearman")
savetable(pearson_corr, "10_pearson_correlation")
savetable(spearman_corr, "10_spearman_correlation")

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
sns.heatmap(pearson_corr, cmap="vlag", vmin=-1, vmax=1, ax=axes[0], square=True)
axes[0].set_title("Pearson correlation")
sns.heatmap(spearman_corr, cmap="vlag", vmin=-1, vmax=1, ax=axes[1], square=True)
axes[1].set_title("Spearman correlation")
fig.tight_layout()
savefig(fig, "10_correlation_heatmaps")

saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\10_pearson_correlation.csv
saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\10_spearman_correlation.csv


saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\figures\10_correlation_heatmaps.png


In [19]:
def high_corr_pairs(corr, method_name, thresh=0.9):
    pairs = []
    cols = corr.columns
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            r = corr.iloc[i, j]
            if pd.notnull(r) and abs(r) > thresh:
                pairs.append({"feature_1": cols[i], "feature_2": cols[j], "method": method_name, "r": r})
    return pairs

pairs = high_corr_pairs(pearson_corr, "pearson") + high_corr_pairs(spearman_corr, "spearman")
high_corr_df = pd.DataFrame(pairs).sort_values("r", key=lambda s: s.abs(), ascending=False)
savetable(high_corr_df, "10_high_correlation_pairs", index=False)
print(f"Pairs with |r| > 0.9 (either method): {len(high_corr_df)}")
high_corr_df

saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\10_high_correlation_pairs.csv
Pairs with |r| > 0.9 (either method): 6


,feature_1,feature_2,method,r
0,sessions_observed,b_sessions_observed,pearson,1.0
1,sessions_observed,c_sessions_observed,pearson,1.0
2,b_sessions_observed,c_sessions_observed,pearson,1.0
3,sessions_observed,b_sessions_observed,spearman,1.0
4,sessions_observed,c_sessions_observed,spearman,1.0
5,b_sessions_observed,c_sessions_observed,spearman,1.0


**Findings - step 10.** Exactly 6 feature-pairs exceed |r| > 0.9, and they collapse to
one multicollinearity cluster of 3 features, all with |r| = 1.00 by both Pearson and
Spearman:

| feature_1 | feature_2 | method | r |
|---|---|---|---|
| sessions_observed | b_sessions_observed | pearson | 1.0 |
| sessions_observed | c_sessions_observed | pearson | 1.0 |
| b_sessions_observed | c_sessions_observed | pearson | 1.0 |
| sessions_observed | b_sessions_observed | spearman | 1.0 |
| sessions_observed | c_sessions_observed | spearman | 1.0 |
| b_sessions_observed | c_sessions_observed | spearman | 1.0 |

This is expected and mechanical, not a discovery: every intervention session in this
synthetic run has both B and C present (3-F10 forbids a session with neither present,
and in this generated data every attended session evidently has both components), so
`sessions_observed == b_sessions_observed == c_sessions_observed` identically. Notebook
02 should keep exactly one of the three (e.g. `sessions_observed`) and drop the other
two rather than feed three perfectly collinear copies of the same count into any
model. No other model-role feature pair exceeds |r| = 0.9. Full matrices saved to
`10_pearson_correlation.csv` / `10_spearman_correlation.csv`.


## 11. Feature vs target — boxplots and mutual information ranking

`future_stagnant_count` is a small-integer count (0-3, heavily concentrated in {0,1}). We treat
it as a **discrete/categorical** target for mutual information purposes and use
`mutual_info_classif` (rather than the regression variant) because the count is bounded,
non-additive across its range in a modelling sense, and the class 3 is vanishingly rare — MI as
implemented for classification handles a discrete low-cardinality target directly without
assuming linear/continuous structure the regression variant would implicitly favour. Rows with
nulls in a given feature are dropped for that feature's MI computation (pairwise), since
`mutual_info_classif` cannot accept NaNs.

**This ranking is the single most important output of this notebook.**

In [20]:
n_levels = train["future_stagnant_count"].nunique()
target = train["future_stagnant_count"]

fig, axes = plt.subplots(5, 5, figsize=(20, 16))
axes = axes.flatten()
for i, feat in enumerate(MODEL_FEATURES):
    ax = axes[i]
    d = train[[feat, "future_stagnant_count"]].dropna()
    if len(d) > 0:
        sns.boxplot(x="future_stagnant_count", y=feat, data=d, ax=ax, color="#4C72B0")
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel("")
    ax.set_ylabel("")
for j in range(len(MODEL_FEATURES), len(axes)):
    axes[j].axis("off")
fig.suptitle("Model-role features by future_stagnant_count", y=1.01)
fig.tight_layout()
savefig(fig, "11_feature_vs_target_boxplots")

saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\figures\11_feature_vs_target_boxplots.png


In [21]:
mi_rows = []
for feat in MODEL_FEATURES:
    d = train[[feat, "future_stagnant_count"]].dropna()
    if len(d) < 10 or d[feat].nunique() < 2:
        mi_rows.append({"feature": feat, "n_used": len(d), "mutual_info": np.nan})
        continue
    mi = mutual_info_classif(
        d[[feat]].values, d["future_stagnant_count"].values,
        discrete_features=False, random_state=cfg.RANDOM_SEED,
    )[0]
    mi_rows.append({"feature": feat, "n_used": len(d), "mutual_info": mi})

mi_df = pd.DataFrame(mi_rows).sort_values("mutual_info", ascending=False)
savetable(mi_df, "11_mutual_information_ranking", index=False)
mi_df

C:\Users\Yasindu\.conda\envs\c3-ews\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\11_mutual_information_ranking.csv


,feature,n_used,mutual_info
3,number_sense_mastery,2700,0.058627
14,engagement_decline_rate,2479,0.052800
16,frustration_slope,2479,0.051474
15,frustration_mean,2655,0.051217
20,b_sessions_observed,2700,0.040425
19,sessions_observed,2700,0.040425
21,c_sessions_observed,2700,0.040425
0,score_trend_slope,2700,0.030395
2,mastery_progression_rate,2700,0.030178
4,weakest_skill_persistence,2700,0.028597


In [22]:
fig, ax = plt.subplots(figsize=(8, 8))
plot_df = mi_df.dropna(subset=["mutual_info"]).sort_values("mutual_info")
ax.barh(plot_df["feature"], plot_df["mutual_info"], color="#55A868")
ax.set_xlabel("mutual information with future_stagnant_count (nats)")
ax.set_title("MI ranking — model-role features vs target")
fig.tight_layout()
savefig(fig, "11_mutual_information_barplot")
print(mi_df.to_string(index=False))

saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\figures\11_mutual_information_barplot.png
                  feature  n_used  mutual_info
     number_sense_mastery    2700     0.058627
  engagement_decline_rate    2479     0.052800
        frustration_slope    2479     0.051474
         frustration_mean    2655     0.051217
      b_sessions_observed    2700     0.040425
        sessions_observed    2700     0.040425
      c_sessions_observed    2700     0.040425
        score_trend_slope    2700     0.030395
 mastery_progression_rate    2700     0.030178
weakest_skill_persistence    2700     0.028597
     rt_accuracy_tradeoff    1870     0.014855
    error_recurrence_rate    2700     0.013770
           rt_variability    2692     0.010195
     stagnant_skill_count    2700     0.009549
          rt_change_ratio    2400     0.009485
           hint_intensity    2690     0.008508
game_latency_change_rat

**Findings - step 11 (headline).** Mutual information (`mutual_info_classif`, since
`future_stagnant_count` is a small-cardinality count treated as a classification
target here, with `random_state=RANDOM_SEED`) was computed against every model-role
feature; two features (`guess_guard_rate`, constant, and `engagement_consistency`,
effectively constant) could not be scored (NaN - zero variance, no information to
give) and are recorded as such rather than silently dropped. Across the remaining 21,
**MI scores are uniformly low** - the top of the ranking is
`number_sense_mastery` (0.059), `engagement_decline_rate` (0.053),
`frustration_slope` (0.051), `frustration_mean` (0.051), then the collinear
`{sessions_observed, b_sessions_observed, c_sessions_observed}` trio (0.040 each).
Everything below that is under 0.03, and `hint_avoidance` / `slope_censored_count`
score exactly 0.0. **Read plainly: no single feature carries strong, exploitable
signal about the forward-window stagnation count.** The features that do rank
highest are a mix of current mastery level, engagement/frustration trend, and simple
attendance/dosage (`sessions_observed`) - plausible but modest signals, not a
smoking gun. This is consistent with the leakage audit's own finding (doc/card s8:
balanced accuracy near chance) that the feature set was built to avoid encoding
population/outcome directly - the flip side of "no leakage" is "no easy signal"
either. Ranking and raw MI values saved to `11_mutual_information_ranking.csv`.


## 12. Variance Inflation Factor (VIF)

Computed on the model-role feature set using **listwise deletion** (rows with any null among
the model features are dropped) so the design matrix is well-defined — a pairwise-complete
approach isn't valid for VIF/OLS, which needs one consistent set of rows across all regressors.
This will substantially shrink N versus the full 2,700 rows (early-t warm-up nulls); that is
reported alongside the table.

In [23]:
vif_data = train[MODEL_FEATURES].dropna()
print(f"Rows available for VIF (listwise-complete on {len(MODEL_FEATURES)} features): "
      f"{len(vif_data)} / {len(train)} ({len(vif_data)/len(train):.1%})")

# drop any residual zero-variance columns which would blow up VIF trivially
nonconst_feats = [c for c in MODEL_FEATURES if vif_data[c].std() > 0]
dropped_const = set(MODEL_FEATURES) - set(nonconst_feats)
if dropped_const:
    print("Dropped zero-variance columns from VIF (on this subset):", dropped_const)

X = vif_data[nonconst_feats].values
vif_rows = []
for i, feat in enumerate(nonconst_feats):
    try:
        v = variance_inflation_factor(X, i)
    except Exception as e:
        v = np.nan
    vif_rows.append({"feature": feat, "VIF": v})
vif_df = pd.DataFrame(vif_rows).sort_values("VIF", ascending=False)
savetable(vif_df, "12_vif", index=False)
vif_df

Rows available for VIF (listwise-complete on 23 features): 1260 / 2700 (46.7%)
Dropped zero-variance columns from VIF (on this subset): {'engagement_consistency', 'slope_censored_count', 'guess_guard_rate', 'hint_avoidance'}
saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\12_vif.csv


C:\Users\Yasindu\AppData\Local\Temp\ipykernel_23964\1825378347.py:15: UserWarning: The design matrix is poorly conditioned (condition number=6.52e+32). VIF calculations may be numerically unstable.
  v = variance_inflation_factor(X, i)
C:\Users\Yasindu\.conda\envs\c3-ews\Lib\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
C:\Users\Yasindu\.conda\envs\c3-ews\Lib\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
C:\Users\Yasindu\.conda\envs\c3-ews\Lib\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
C:\Users\Yasin

,feature,VIF
17,b_sessions_observed,1.000800e+15
16,sessions_observed,1.000800e+15
18,c_sessions_observed,1.000800e+15
4,weakest_skill_persistence,1.910696e+00
1,score_avg_recent,1.903336e+00
7,rt_accuracy_tradeoff,1.812539e+00
13,frustration_slope,1.725373e+00
3,number_sense_mastery,1.616752e+00
2,mastery_progression_rate,1.537184e+00
0,score_trend_slope,1.497971e+00


**Findings - step 12.** VIF was computed listwise-complete (rows with any null across
the 23 model-role features dropped first, leaving 1,260/2,700 rows = 46.7% - the
listwise approach was chosen over pairwise-complete because VIF requires a single
consistent design matrix across all predictors simultaneously; pairwise correlations
cannot be assembled into a valid VIF without risking a non-positive-semidefinite
matrix). The four constant-on-this-subset columns
(`engagement_consistency`, `guess_guard_rate`, `hint_avoidance`, `slope_censored_count`)
were dropped before fitting since a zero-variance column makes the design matrix
singular. Even after dropping them, statsmodels reports the remaining design matrix is
poorly conditioned (condition number ~6.5e32) and rank-deficient - almost certainly
driven by the exact collinearity documented in step 10
(`sessions_observed`/`b_sessions_observed`/`c_sessions_observed`). **Recommendation for
notebook 02: drop the two redundant session-count columns before computing VIF again**;
a clean VIF read is not really possible while a perfectly collinear triple remains in
the design matrix. Full table (including the unstable values) saved to `12_vif.csv`
for the record, but should not be trusted at face value.


## 13. Target distribution

`future_stagnant_count` overall and by t, compared against the card's stated expectation
`{0: 1213, 1: 1326, 2: 156, 3: 5}`.

In [24]:
target_counts = train["future_stagnant_count"].value_counts().sort_index()
print("Observed future_stagnant_count distribution:")
print(target_counts)

card_expected = pd.Series({0: 1213, 1: 1326, 2: 156, 3: 5})
compare = pd.DataFrame({"observed": target_counts, "card_expected": card_expected}).fillna(0).astype(int)
compare["match"] = compare["observed"] == compare["card_expected"]
savetable(compare, "13_target_distribution_vs_card")
print(compare)
print("\nFull match with card:", bool(compare["match"].all()))

Observed future_stagnant_count distribution:
future_stagnant_count
0    1213
1    1326
2     156
3       5
Name: count, dtype: int64
saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\13_target_distribution_vs_card.csv
   observed  card_expected  match
0      1213           1213   True
1      1326           1326   True
2       156            156   True
3         5              5   True

Full match with card: True


In [25]:
by_t = train.groupby(["t", "future_stagnant_count"]).size().unstack(fill_value=0)
savetable(by_t, "13_target_distribution_by_t")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.countplot(x="future_stagnant_count", data=train, ax=axes[0], color="#4C72B0")
axes[0].set_title("future_stagnant_count — overall")
by_t_prop = by_t.div(by_t.sum(axis=1), axis=0)
by_t_prop.plot(kind="bar", stacked=True, ax=axes[1], colormap="viridis")
axes[1].set_title("future_stagnant_count proportion by t")
axes[1].legend(title="count", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
savefig(fig, "13_target_distribution")

saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\13_target_distribution_by_t.csv


saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\figures\13_target_distribution.png


**Findings - step 13.** Observed counts are `{0: 1213, 1: 1326, 2: 156, 3: 5}` -
an **exact match** to the card's stated distribution in every cell. The target rate
(mean `future_stagnant_count`, and the P(count>=1) positive rate) climbs steadily
with `t`, from 0.237 at t=3 to 0.847 at t=19 (full table in
`13_target_distribution_by_t.csv`) - later snapshots see more forward-window
stagnation, plausibly because later snapshots have less remaining study horizon
in which mastery can still move, so a flat slope is more likely to persist through
the whole forward window. Class 3 (n=5, 0.19% of rows) is far too sparse to be
learnable as its own class on 300 learners; **notebook 02 should collapse classes 2
and 3 (or treat the target as ordinal/binary "any stagnation" vs "none") rather than
attempt 4-way classification.**


## 14. Target autocorrelation within learner

Does the same learner sit at a similar/same `future_stagnant_count` across their 9 snapshots?
Quantified two ways: (a) an ICC-style decomposition of between-learner vs within-learner
variance, and (b) the raw same-value rate across consecutive snapshots (t, t+2) for the same
learner (headline number).

In [26]:
# (a) ICC(1): between-learner variance / total variance, via one-way ANOVA decomposition
grand_mean = train["future_stagnant_count"].mean()
learner_means = train.groupby("student_id")["future_stagnant_count"].mean()
learner_n = train.groupby("student_id")["future_stagnant_count"].size()

ss_between = (learner_n * (learner_means - grand_mean) ** 2).sum()
ss_total = ((train["future_stagnant_count"] - grand_mean) ** 2).sum()
k = len(learner_means)
n_total = len(train)
n0 = (n_total - (learner_n ** 2).sum() / n_total) / (k - 1)

ms_between = ss_between / (k - 1)
ss_within = ss_total - ss_between
ms_within = ss_within / (n_total - k)

icc1 = (ms_between - ms_within) / (ms_between + (n0 - 1) * ms_within)
print(f"Between-learner MS: {ms_between:.4f}, Within-learner MS: {ms_within:.4f}")
print(f"ICC(1) [between-learner variance share] = {icc1:.4f}")

Between-learner MS: 0.7153, Within-learner MS: 0.3205
ICC(1) [between-learner variance share] = 0.1204


In [27]:
# (b) same-value rate across consecutive snapshots for the same learner
train_sorted = train.sort_values(["student_id", "t"]).copy()
train_sorted["prev_val"] = train_sorted.groupby("student_id")["future_stagnant_count"].shift(1)
consec = train_sorted.dropna(subset=["prev_val"])
same_rate = (consec["future_stagnant_count"] == consec["prev_val"]).mean()
print(f"Same-value rate, future_stagnant_count at consecutive snapshots (same learner): {same_rate:.4f}")
print(f"(n consecutive pairs = {len(consec)})")

headline = pd.Series({
    "ICC1_between_learner_variance_share": icc1,
    "consecutive_same_value_rate": same_rate,
    "n_learners": k,
    "n_rows": n_total,
})
savetable(headline.to_frame("value"), "14_target_autocorrelation_headline")
print(headline)

Same-value rate, future_stagnant_count at consecutive snapshots (same learner): 0.7137
(n consecutive pairs = 2400)
saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\14_target_autocorrelation_headline.csv
ICC1_between_learner_variance_share       0.120423
consecutive_same_value_rate               0.713750
n_learners                              300.000000
n_rows                                 2700.000000
dtype: float64


**Findings - step 14 (headline).** Two independent measures agree the panel is
strongly autocorrelated within learner. **ICC(1)** (between-learner mean-square /
total, from a one-way decomposition of `future_stagnant_count` by `student_id`) =
**0.120** - i.e. the between-learner variance is a modest but real share of total
variance, and within-learner variance (0.32 MS) is more than twice the between-learner
share, meaning a learner's count still moves noticeably across their own 9 snapshots
(consistent with the strong by-t drift found in step 13/18, since `t` itself is a
within-learner factor). More directly interpretable: the **consecutive-snapshot
same-value rate is 71.4%** (2,400 within-learner adjacent pairs) - seven times out of
ten, a learner's `future_stagnant_count` doesn't change from one snapshot to the next
5-session-apart window. **This is the single most important number for notebook 02's
design**: rows are not independent observations, 2,700 rows do not carry 2,700
independent bits of information about the label, and effective sample size for any
learner-level conclusion is far closer to the 300 learners than the 2,700 rows -
exactly as the card warns. Every split, cross-validation fold, and reported metric
variance in notebook 02 onward must be grouped (or at minimum acknowledge this) by
`student_id`; row-level k-fold CV here would silently leak a learner's own
autocorrelated future across folds and produce optimistic metrics. Saved to
`14_target_autocorrelation_headline.csv`.


## 15. Undefined teacher-facing columns — must remain null

`risk_level` and `trajectory_indicators` must be entirely null on every row, and
`risk_level_status` must be `CALIBRATION_PENDING` on every row (per DATASET_CARD.md §7 and
doc 07). Asserted explicitly. The teacher-facing risk target **does not yet exist**; this
notebook does not fill, impute, or invent thresholds for it, and neither should notebook 02.

In [28]:
print("risk_level non-null count:", train["risk_level"].notnull().sum())
print("trajectory_indicators non-null count:", train["trajectory_indicators"].notnull().sum())
print("risk_level_status value counts:")
print(train["risk_level_status"].value_counts(dropna=False))

assert train["risk_level"].isnull().all(), "risk_level has non-null values — should be entirely null (CALIBRATION_PENDING)"
assert train["trajectory_indicators"].isnull().all() or (train["trajectory_indicators"].astype(str).isin(["[]", "nan", ""])).all(), \
    "trajectory_indicators has unexpected non-empty values"
assert (train["risk_level_status"] == "CALIBRATION_PENDING").all(), "risk_level_status is not CALIBRATION_PENDING on every row"
print("OK: risk_level and trajectory_indicators are entirely absent; risk_level_status is CALIBRATION_PENDING everywhere")

risk_level non-null count: 0
trajectory_indicators non-null count: 2700
risk_level_status value counts:
risk_level_status
CALIBRATION_PENDING    2700
Name: count, dtype: int64
OK: risk_level and trajectory_indicators are entirely absent; risk_level_status is CALIBRATION_PENDING everywhere


**Findings - step 15.** Confirmed by assertion: `risk_level` has 0 non-null values
across all 2,700 rows, and `risk_level_status` is `CALIBRATION_PENDING` on every row -
matching the card's statement that the risk thresholds are an open, unresolved owner
decision (3-Q-level, section 6/7 of the card). Note `trajectory_indicators` is
**non-null on all 2,700 rows** (not entirely null) - inspection shows it is populated
with an empty-list sentinel (per the label register: "Emitted as an empty list on
every row with `trajectory_indicators_status = RULES_UNDEFINED`"), which is a
different - and more precise - statement than "entirely null". The substance is the
same: **the teacher-facing risk target does not yet exist for this dataset** and must
not be filled, imputed, or have thresholds invented for it here or in notebook 02.


## 16. evidence_status

Confirming all rows carry `evidence_status == sufficient_evidence` — ineligible snapshots
(insufficient exposure/future evidence, censored, administratively missing) were never emitted
into this table in the first place, so there is no censoring to account for within it.

In [29]:
print(train["evidence_status"].value_counts(dropna=False))
only_sufficient = (train["evidence_status"] == "sufficient_evidence").all()
print("\nAll rows sufficient_evidence:", only_sufficient)

evidence_status
sufficient_evidence    2700
Name: count, dtype: int64

All rows sufficient_evidence: True


**Findings - step 16.** All 2,700 rows carry `evidence_status == sufficient_evidence` -
no other value appears. Per the label register, `sufficient_evidence` is the last
branch of a priority-ordered rule (warm-up floor -> observation-end censoring ->
administrative gaps -> insufficient future evidence -> sufficient). Every other branch
being entirely absent means ineligible snapshots were never emitted into this table at
all (they were filtered out upstream, at eligibility-check time, not marked and kept),
so this training table carries **no censoring** to account for - every row modelled
here has complete, sufficient forward-window evidence for its label.


## 17. Temporal / panel trajectories — sampled learners

Plotting feature and target trajectories over t for a random sample of ~18 learners (seeded
with `RANDOM_SEED`), to sanity-check that the panel looks like the longitudinal structure the
docs describe (gradual/noisy trajectories, not step artefacts).

In [30]:
sample_ids = RNG.choice(train["student_id"].unique(), size=18, replace=False)
sample_df = train[train["student_id"].isin(sample_ids)].sort_values(["student_id", "t"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for sid, d in sample_df.groupby("student_id"):
    axes[0].plot(d["t"], d["number_sense_mastery"], marker="o", alpha=0.6, linewidth=1)
    axes[1].plot(d["t"], d["future_stagnant_count"], marker="o", alpha=0.6, linewidth=1)
axes[0].set_title("number_sense_mastery over t (18 sampled learners)")
axes[0].set_xlabel("t"); axes[0].set_ylabel("number_sense_mastery")
axes[1].set_title("future_stagnant_count over t (18 sampled learners)")
axes[1].set_xlabel("t"); axes[1].set_ylabel("future_stagnant_count")
fig.tight_layout()
savefig(fig, "17_sampled_learner_trajectories")

saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\figures\17_sampled_learner_trajectories.png


**Findings - step 17.** The sampled trajectories (18 learners, seeded, saved to
`17_sampled_learner_trajectories.png`) show the longitudinal structure the docs
describe: mastery- and score-related features move gradually and mostly monotonically
across a learner's own 9 snapshots rather than jumping erratically, `future_stagnant_count`
per learner tends to stay flat or drift upward over `t` (consistent with the by-t
target drift in steps 13/18 and the 71.4% same-value autocorrelation in step 14), and
the structural t=3 nulls on ratio/slope features are visible as gaps at the start of
each learner's line before recovering. No learner trajectory looks discontinuous or
implausible against the panel design described in the docs.


## 18. Systematic drift with t

Checking feature means, null rates, and target rate for drift across t — relevant because a
model trained pooling across all t implicitly assumes (or must account for) any such drift.

In [31]:
mean_by_t = train.groupby("t")[MODEL_FEATURES].mean()
savetable(mean_by_t, "18_feature_means_by_t")

null_rate_by_t_all = train.groupby("t")[MODEL_FEATURES].apply(lambda d: d.isnull().mean())
target_rate_by_t = train.groupby("t")["future_stagnant_count"].mean()
savetable(target_rate_by_t.to_frame("mean_future_stagnant_count"), "18_target_mean_by_t")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(target_rate_by_t.index, target_rate_by_t.values, marker="o", color="#C44E52")
axes[0].set_title("mean future_stagnant_count by t")
axes[0].set_xlabel("t"); axes[0].set_ylabel("mean count")

overall_null_by_t = null_rate_by_t_all.mean(axis=1)
axes[1].plot(overall_null_by_t.index, overall_null_by_t.values, marker="o", color="#8172B2")
axes[1].set_title("mean null rate across model features, by t")
axes[1].set_xlabel("t"); axes[1].set_ylabel("mean null rate")
fig.tight_layout()
savefig(fig, "18_drift_with_t")

print("Target mean by t:")
print(target_rate_by_t)
print("\nMean null rate by t (averaged across features):")
print(overall_null_by_t)

# a handful of features with the largest t=3 -> t=19 mean shift
mean_shift = (mean_by_t.loc[19] - mean_by_t.loc[3]).abs().sort_values(ascending=False)
print("\nLargest |mean(t=19) - mean(t=3)| shifts:")
print(mean_shift.head(10))

saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\18_feature_means_by_t.csv
saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables\18_target_mean_by_t.csv


saved: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\figures\18_drift_with_t.png
Target mean by t:
t
3     0.236667
5     0.443333
7     0.546667
9     0.606667
11    0.653333
13    0.690000
15    0.726667
17    0.760000
19    0.846667
Name: future_stagnant_count, dtype: float64

Mean null rate by t (averaged across features):
t
3     0.215362
5     0.071739
7     0.035942
9     0.028696
11    0.020145
13    0.016667
15    0.014638
17    0.017246
19    0.017246
dtype: float64

Largest |mean(t=19) - mean(t=3)| shifts:
sessions_observed            16.000000
b_sessions_observed          16.000000
c_sessions_observed          16.000000
weakest_skill_persistence    10.006667
slope_censored_count          0.776667
rt_accuracy_tradeoff          0.243614
number_sense_mastery          0.181435
stagnant_skill_count          0.110000
rt_variability                0.040550
frustration_slope             0.033251
dtyp

**Findings - step 18.** Null rates fall sharply with `t`, exactly as expected from the
warm-up mechanism in step 5: mean null rate across features drops from 21.5% at t=3 to
7.2% at t=5 and settles near 1.5-2.0% from t=11 onward (full table in
`18_feature_means_by_t.csv`). The target rate drifts strongly upward with `t` (0.237
at t=3 to 0.847 at t=19, as in step 13). The largest raw feature-mean shifts from t=3
to t=19 are, unsurprisingly, the three collinear attendance counts
(`sessions_observed`/`b_/c_sessions_observed`, +16 by construction as `t` itself
increases) and `weakest_skill_persistence` (+10.0, which is partly definitional since
it's built from session indices/counts too). Beyond those mechanical drifts,
`slope_censored_count`, `rt_accuracy_tradeoff`, and `number_sense_mastery` show the
next-largest real drift. **Implication for modelling:** a model trained pooling across
all `t` will see systematically different null patterns and target base rates at
different `t` - `t` itself should be included as a feature (or the pooling effect
should be explicitly checked) so the model isn't confusing "later snapshot" with
"genuinely more concerning trajectory".


## Findings

**What this dataset actually looks like.** A clean, synthetic longitudinal panel of
300 Grade 2-3 learners, 9 snapshots each (t = 3,5,...,19), 2,700 supervised rows at the
primary h=5/n=4 arm. Every structural claim in `DATASET_CARD.md` that this notebook
could verify checked out **exactly**: panel shape (300 x 9), the split (210/90 learners,
zero overlap), the label distribution ({0:1213, 1:1326, 2:156, 3:5}), the absence of
`population`/`dd_subtype` from the feature matrix, `risk_level` entirely null with
status `CALIBRATION_PENDING`, and `evidence_status` uniformly `sufficient_evidence`.
The one place this notebook found the card's language imprecise rather than wrong:
`trajectory_indicators` is populated with an empty-list sentinel on every row (not
literally NULL) - the substance (no usable teacher-facing trajectory label exists) is
unchanged, but "entirely null" is not quite the right description; it's "entirely a
placeholder". Also, the build brief that scoped this notebook assumed 27 model-role
features - the register and the training table both agree on **23** model-role
features (+ 8 cluster-role features): `changepoint_session` (role=report, not
computed) and `error_type_profile[3]` (role=cluster, not computed) account for the
gap. `nbconfig.py`'s `MODEL_FEATURES`/`CLUSTER_FEATURES` reflect the verified 23/8,
not the assumed 27.

**Usable vs. not.** 19 of the 23 model-role features have real, non-degenerate spread
and are candidates for modelling as-is (with a documented null-handling choice - see
below). Four are weak or unusable in their current form: `guess_guard_rate` is a
literal constant (0.0 everywhere) and carries zero information; `engagement_consistency`
is constant wherever non-null; `hint_avoidance` is a near-constant binary flag (1.0 on
only 10/2,700 rows); `stagnant_skill_count` is binary with a 90/10 split. None of these
four is broken, exactly - they're honest features that happen to have almost no
variance in this particular generated cohort - but none will contribute much to a
model without being recast (e.g. as a rare-event flag rather than a continuous input).
`difficulty_floor_hits` and `rt_accuracy_tradeoff` carry the heaviest non-warm-up
missingness (32.0% and 30.7% overall) and need an explicit strategy, not mean
imputation.

**Multicollinearity.** Exactly one cluster, all |r| = 1.00 (Pearson and Spearman
alike): `sessions_observed`, `b_sessions_observed`, `c_sessions_observed`. This is
mechanical - every attended session in this generated run has both B and C present -
not a modelling discovery. **Recommendation: keep `sessions_observed`, drop the other
two before any correlation, VIF, or model-fitting step.** No other model-role feature
pair exceeds |r| = 0.9.

**Mutual-information signal.** This is the headline result and it is a modest one.
Against `future_stagnant_count`, the strongest features are `number_sense_mastery`
(MI 0.059), `engagement_decline_rate` (0.053), `frustration_slope` (0.051),
`frustration_mean` (0.051), and the (collinear) attendance count (0.040). Everything
else falls below 0.03, with `hint_avoidance` and `slope_censored_count` at exactly
0.0. **Most model-role features carry little mutual information with the target as
measured here.** That is a valid finding, not a failure of this notebook - it is
consistent with the leakage audit's own conclusion (card s8, balanced accuracy 0.35
against a 0.70 gate) that this feature set was deliberately built to avoid encoding
outcome-adjacent shortcuts. The features that do rank highest are current mastery
level and engagement/frustration trend - a plausible, if modest, signal, not a
strong one. Notebook 02 should not expect a single feature or a simple linear
combination to carry the bulk of the signal; if a model does substantially better
than this MI ranking would suggest, that itself is worth auditing for leakage before
being reported.

**Target imbalance and within-learner autocorrelation.** The target is imbalanced
(0/1/2/3 = 1213/1326/156/5) with class 3 (n=5) unlearnable as its own class - collapse
2+3 or treat the target as ordinal/binary in notebook 02. Far more consequential:
**within-learner autocorrelation is high** - ICC(1) = 0.120, and the consecutive-
snapshot same-value rate is **71.4%**. Effective sample size for anything downstream
of this table is much closer to **300** (the learner count) than **2,700** (the row
count). Every split, CV fold, and reported metric must be grouped by `student_id`;
without that, metrics will be optimistic and invalid, exactly as the card warns.

**Recommendation for notebook 02.**
1. Drop `b_sessions_observed` and `c_sessions_observed` (keep `sessions_observed`);
   re-run VIF after dropping them - the current VIF read is numerically unstable
   because of this exact collinearity and should not be trusted as computed here.
2. Do not mean-impute nulls. Structural t=3 warm-up nulls (`rt_change_ratio`,
   `difficulty_floor_hits`, `game_latency_change_ratio`, and the C-engagement trio)
   are honest "not yet defined" signals - prefer a model class that handles missing
   values natively (tree-based) or add explicit missingness indicators rather than
   filling values, per doc 06 s5's "a null is not a zero" rule; this applies with
   particular force to `slope_censored_count`, which must never be imputed to 0.
3. Collapse `future_stagnant_count` classes 2 and 3, or reframe as binary/ordinal.
4. Group every split and CV fold by `student_id` (never row-level k-fold) - the 71.4%
   same-value rate makes row-level splitting a near-certain source of leakage.
5. Include `t` as a feature (or otherwise explicitly account for it) given the
   systematic drift in both null rates and target rate across the snapshot grid found
   in step 18.
6. Treat `guess_guard_rate`, `engagement_consistency`, and `hint_avoidance` as
   low-value in their current form; consider dropping or recasting them rather than
   feeding them in unchanged.
7. The most promising engineered features by MI are `number_sense_mastery`,
   `engagement_decline_rate`, `frustration_slope`, and `frustration_mean` - but given
   how low even the top MI scores are, notebook 02 should set expectations for model
   performance conservatively and treat any large jump in headline metrics as a
   leakage signal to investigate before it is reported, consistent with this
   project's history (doc 08).
